# 🔥 Day 3 — 2D Heat Equation PINN

---

## 📌 What changes from Day 2?

| | Day 2 (1D) | Day 3 (2D) |
|---|---|---|
| Network input | `[x, t]` | `[x, y, t]` |
| PDE | `∂u/∂t = α ∂²u/∂x²` | `∂u/∂t = α(∂²u/∂x² + ∂²u/∂y²)` |
| Boundaries | 2 endpoints | 4 walls |
| Network size | 4 × 32 | 4 × 64 (wider) |
| Visualization | Line plots | 2D heatmaps |

## 📌 Problem Setup

$$\frac{\partial u}{\partial t} = \alpha \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right)$$

| | Value |
|---|---|
| α | 0.01 |
| Domain | x,y ∈ [0,1], t ∈ [0,1] |
| IC | u(x,y,0) = sin(πx)·sin(πy) |
| BC | u = 0 on all 4 walls |
| Exact solution | exp(−2α π² t)·sin(πx)·sin(πy) |

---
## 🔹 Step 1: Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os

torch.manual_seed(42)
os.makedirs('outputs', exist_ok=True)

ALPHA = 0.01

def exact(x, y, t):
    return (torch.exp(-2 * ALPHA * np.pi**2 * t)
            * torch.sin(np.pi * x)
            * torch.sin(np.pi * y))

print('✅ Setup done')
print('PDE: du/dt = alpha*(d2u/dx2 + d2u/dy2)')
print(f'alpha = {ALPHA}')

---
## 🔹 Step 2: Network

Now **3 inputs** instead of 2 — x, y, and t stacked together.
Wider hidden layers (64) because 2D is harder to learn than 1D.

In [ ]:
class PINN2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),  nn.Tanh(),   # 3 inputs: x, y, t
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)               # output: temperature u
        )

    def forward(self, x, y, t):
        inp = torch.cat([x, y, t], dim=1)  # shape (N, 3)
        return self.net(inp)                # shape (N, 1)

model = PINN2D()
params = sum(p.numel() for p in model.parameters())
print(f'✅ Network ready | Parameters: {params}')
print(model)

---
## 🔹 Step 3: Collocation Point Samplers

4 walls now instead of 2 endpoints — one for each edge of the square.

In [ ]:
def interior(N=3000):
    """random (x,y,t) inside domain — physics loss here"""
    x = torch.rand(N, 1, requires_grad=True)
    y = torch.rand(N, 1, requires_grad=True)
    t = torch.rand(N, 1, requires_grad=True)
    return x, y, t

def ic_points(N=500):
    """t=0 points — u(x,y,0) = sin(pi*x)*sin(pi*y)"""
    x = torch.rand(N, 1)
    y = torch.rand(N, 1)
    t = torch.zeros(N, 1)
    u_true = torch.sin(np.pi * x) * torch.sin(np.pi * y)
    return x, y, t, u_true

def bc_points(N=200):
    """4 walls: x=0, x=1, y=0, y=1 — all u=0"""
    t = torch.rand(N, 1)
    s = torch.rand(N, 1)   # position along the wall
    # left and right walls (x fixed)
    x_l = torch.zeros(N, 1); y_l = s.clone()
    x_r = torch.ones(N, 1);  y_r = s.clone()
    # bottom and top walls (y fixed)
    x_b = s.clone(); y_b = torch.zeros(N, 1)
    x_t = s.clone(); y_t = torch.ones(N, 1)
    return (x_l,y_l), (x_r,y_r), (x_b,y_b), (x_t,y_t), t

print('✅ Samplers ready')
print('  Interior : 3000 points (physics loss)')
print('  IC       : 500 points  (initial condition)')
print('  BC       : 200 points × 4 walls (boundary condition)')

---
## 🔹 Step 4: Physics Loss — 2D Version

Now we need **both** ∂²u/∂x² **and** ∂²u/∂y². Their sum is the **Laplacian** ∇²u.

$$r = \frac{\partial u}{\partial t} - \alpha\left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}\right)$$

In [ ]:
def physics_loss(model, x, y, t):
    u = model(x, y, t)

    # du/dt
    du_dt = torch.autograd.grad(
        u, t, torch.ones_like(u), create_graph=True)[0]

    # d2u/dx2
    du_dx = torch.autograd.grad(
        u, x, torch.ones_like(u), create_graph=True)[0]
    d2u_dx2 = torch.autograd.grad(
        du_dx, x, torch.ones_like(du_dx), create_graph=True)[0]

    # d2u/dy2  ← new compared to day 2
    du_dy = torch.autograd.grad(
        u, y, torch.ones_like(u), create_graph=True)[0]
    d2u_dy2 = torch.autograd.grad(
        du_dy, y, torch.ones_like(du_dy), create_graph=True)[0]

    # laplacian = d2u/dx2 + d2u/dy2
    laplacian = d2u_dx2 + d2u_dy2
    r = du_dt - ALPHA * laplacian
    return torch.mean(r ** 2)

print('✅ Physics loss ready')
print('   Laplacian = d2u/dx2 + d2u/dy2')
print('   Residual  = du/dt - alpha * Laplacian')

---
## 🔹 Step 5: Training Loop

Same structure as Day 2 — just more epochs because 2D is harder.

> ⏱️ ~5 min on Colab GPU, ~15 min on CPU

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
EPOCHS = 8000

print(f"Training for {EPOCHS} epochs...")
print(f"{'Epoch':<8} {'Total':<12} {'Physics':<12} {'IC':<12} {'BC':<10}")
print("-" * 54)

for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad()

    # physics loss
    x_i, y_i, t_i = interior(3000)
    loss_pde = physics_loss(model, x_i, y_i, t_i)

    # ic loss
    x_ic, y_ic, t_ic, u_ic = ic_points(500)
    loss_ic = torch.mean((model(x_ic, y_ic, t_ic) - u_ic) ** 2)

    # bc loss — all 4 walls
    walls_data = bc_points(200)
    walls, t_bc = walls_data[:-1], walls_data[-1]
    loss_bc = sum(torch.mean(model(xb, yb, t_bc) ** 2)
                  for xb, yb in walls)

    loss = loss_pde + 10 * loss_ic + 10 * loss_bc
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 1000 == 0:
        print(f"{epoch:<8} {loss.item():<12.5f} {loss_pde.item():<12.5f} "
              f"{loss_ic.item():<12.5f} {loss_bc.item():<10.5f}")

print('\n✅ Training complete!')

---
## 🔹 Step 6: Visualize — 2D Temperature Maps

In [ ]:
model.eval()
n = 50
x1d = np.linspace(0, 1, n)
y1d = np.linspace(0, 1, n)
X2d, Y2d = np.meshgrid(x1d, y1d)

t_snapshots = [0.0, 0.1, 0.3, 0.6]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('2D Heat Equation PINN\ndu/dt = alpha*(d2u/dx2 + d2u/dy2)',
             fontsize=13, fontweight='bold')

for col, ti in enumerate(t_snapshots):
    xf = torch.tensor(X2d.flatten(), dtype=torch.float32).unsqueeze(1)
    yf = torch.tensor(Y2d.flatten(), dtype=torch.float32).unsqueeze(1)
    tf = torch.full((n*n, 1), ti)

    with torch.no_grad():
        u_pred = model(xf, yf, tf).numpy().reshape(n, n)

    u_true = (np.exp(-2*ALPHA*np.pi**2*ti)
              * np.sin(np.pi*X2d) * np.sin(np.pi*Y2d))

    # pinn row
    ax = axes[0, col]
    im = ax.contourf(X2d, Y2d, u_pred, levels=30, cmap='hot', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_title(f'PINN  t={ti}')
    ax.set_xlabel('x'); ax.set_ylabel('y')

    # exact row
    ax = axes[1, col]
    im2 = ax.contourf(X2d, Y2d, u_true, levels=30, cmap='hot', vmin=0, vmax=1)
    plt.colorbar(im2, ax=ax)
    ax.set_title(f'Exact  t={ti}')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.savefig('outputs/day3_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Plot saved to outputs/day3_result.png')

---
## 🔹 Step 7: Accuracy Check

In [ ]:
# check at t=0.6 (middle of time domain)
xf = torch.tensor(X2d.flatten(), dtype=torch.float32).unsqueeze(1)
yf = torch.tensor(Y2d.flatten(), dtype=torch.float32).unsqueeze(1)
tf = torch.full((n*n, 1), 0.6)

with torch.no_grad():
    u_pred_f = model(xf, yf, tf).numpy().reshape(n, n)

u_true_f = (np.exp(-2*ALPHA*np.pi**2*0.6)
            * np.sin(np.pi*X2d) * np.sin(np.pi*Y2d))

rel_err = np.linalg.norm(u_pred_f - u_true_f) / np.linalg.norm(u_true_f)
max_err = np.abs(u_pred_f - u_true_f).max()

print(f'📊 Accuracy at t=0.6')
print(f'   Relative L2 error : {rel_err*100:.3f}%')
print(f'   Max absolute error: {max_err:.5f}')

# loss curve
plt.figure(figsize=(8,4))
plt.semilogy(losses, color='#e63946', lw=1.5)
plt.xlabel('Epoch'); plt.ylabel('Loss (log)')
plt.title('Training Loss — 2D Heat Equation')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## ✅ Day 3 Summary

| | Day 2 | Day 3 |
|---|---|---|
| Input | `[x, t]` | `[x, y, t]` |
| PDE terms | 1 second derivative | 2 second derivatives (Laplacian) |
| Boundaries | 2 | 4 walls |
| Network | 4×32 | 4×64 |
| Epochs | 5000 | 8000 |

---
### 🚀 Next: Day 4 — Burgers Equation (Nonlinear)
The real benchmark. Nonlinear PDE, shock waves, the exact problem from the original PINN paper.

Say **"Day 4"** when ready 🔥